In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

In [ ]:
# --- 1) Daten laden ---

drl_data = pd.read_csv(
    "data/data_2019-01-01_2024-01-01_hourly.csv",
    parse_dates=["time"],
    index_col="time",
)
drl_data.index = pd.to_datetime(drl_data.index, utc=True)

ri_profit1 = pd.read_csv(
    "output/single_market/rolling_intrinsic/ri_basic/qh/2019/bs15cr1rto0.86mc365mt10/profit.csv"
)
ri_profit2 = pd.read_csv(
    "output/single_market/rolling_intrinsic/ri_basic/qh/2023/bs15cr1rto0.86mc365mt10/profit.csv"
)
ri_profit = pd.concat([ri_profit1, ri_profit2], ignore_index=True)

ri_profit["day"] = pd.to_datetime(ri_profit["day"], errors="coerce", utc=True,)
ri_profit["day"] = ri_profit["day"].dt.tz_convert("Europe/Berlin")
ri_profit = ri_profit.sort_values("day").reset_index(drop=True)
ri_profit = ri_profit.set_index("day").sort_index()
ri_profit.index = ri_profit.index.tz_convert("UTC")


"""
# WICHTIG: hier nur to_datetime mit utc=True, KEIN tz_localize
ri_profit["day"] = pd.to_datetime(ri_profit["day"], utc=True)

ri_profit = ri_profit.set_index("day").sort_index()
ri_profit.index.name = "date"
"""

'\n# WICHTIG: hier nur to_datetime mit utc=True, KEIN tz_localize\nri_profit["day"] = pd.to_datetime(ri_profit["day"], utc=True)\n\nri_profit = ri_profit.set_index("day").sort_index()\nri_profit.index.name = "date"\n'

In [ ]:
ri_profit

,profit,cycles
day,,
2018-12-31 23:00:00+00:00,68.233139,1.0
2019-01-01 23:00:00+00:00,159.440419,2.0
2019-01-02 23:00:00+00:00,56.598649,3.0
2019-01-03 23:00:00+00:00,64.915423,4.0
2019-01-04 23:00:00+00:00,102.020127,5.0
...,...,...
2023-12-26 23:00:00+00:00,106.523683,88.0
2023-12-27 23:00:00+00:00,102.341338,89.0
2023-12-28 23:00:00+00:00,91.073262,90.0


In [ ]:
# --- 2) Residual Load berechnen ---

drl_data["residual_load"] = (
    drl_data["load_forecast_d_minus_1_1000_total_de_lu_mw"]
    - drl_data["pv_forecast_d_minus_1_1000_de_lu_mw"]
    - drl_data["wind_offshore_forecast_d_minus_1_1000_de_lu_mw"]
    - drl_data["wind_onshore_forecast_d_minus_1_1000_de_lu_mw"]
)

In [ ]:
# --- 3) Daily Aggregation über Resample ---

agg_features = [
    "epex_spot_60min_de_lu_eur_per_mwh",
    "exaa_15min_de_lu_eur_per_mwh",
    "load_forecast_d_minus_1_1000_total_de_lu_mw",
    "pv_forecast_d_minus_1_1000_de_lu_mw",
    "wind_offshore_forecast_d_minus_1_1000_de_lu_mw",
    "wind_onshore_forecast_d_minus_1_1000_de_lu_mw",
    "residual_load",
    "id_full_h",
    "id_full_qh",
]

agg_dict = {col: ["mean", "std", "min", "max"] for col in agg_features}

daily_agg = (
    drl_data
    .resample("D")      # täglich auf Index 'time'
    .agg(agg_dict)
)

daily_agg.columns = [
    f"{col}_{stat}" for col, stat in daily_agg.columns.to_flat_index()
]
daily_agg.index.name = "date"

daily_agg["epex_spread"] = (
    daily_agg["epex_spot_60min_de_lu_eur_per_mwh_max"]
    - daily_agg["epex_spot_60min_de_lu_eur_per_mwh_min"]
)

daily_agg["residual_load_spread"] = (
    daily_agg["residual_load_max"] - daily_agg["residual_load_min"]
)

daily_agg["day_of_week"] = daily_agg.index.dayofweek
daily_agg["month"] = daily_agg.index.month
daily_agg["year"] = daily_agg.index.year


df_daily = daily_agg.join(ri_profit[["profit", "cycles"]], how="inner")
print(df_daily.shape)

(1826, 43)


In [ ]:
# --- 4) Features + Target zusammenführen ---

# Annahme: ri_daily hat Spalten 'profit' und 'cycles'
ri_profit.index.name = "date"
df_daily = daily_agg.merge(
    ri_profit[["profit", "cycles"]],
    left_index=True,
    right_index=True,
    how="inner"
)

In [ ]:
daily_agg.index

DatetimeIndex(['2018-12-31 00:00:00+00:00', '2019-01-01 00:00:00+00:00',
               '2019-01-02 00:00:00+00:00', '2019-01-03 00:00:00+00:00',
               '2019-01-04 00:00:00+00:00', '2019-01-05 00:00:00+00:00',
               '2019-01-06 00:00:00+00:00', '2019-01-07 00:00:00+00:00',
               '2019-01-08 00:00:00+00:00', '2019-01-09 00:00:00+00:00',
               ...
               '2023-12-22 00:00:00+00:00', '2023-12-23 00:00:00+00:00',
               '2023-12-24 00:00:00+00:00', '2023-12-25 00:00:00+00:00',
               '2023-12-26 00:00:00+00:00', '2023-12-27 00:00:00+00:00',
               '2023-12-28 00:00:00+00:00', '2023-12-29 00:00:00+00:00',
               '2023-12-30 00:00:00+00:00', '2023-12-31 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='date', length=1827, freq='D')

In [ ]:
ri_profit.index

DatetimeIndex(['2018-12-31 00:00:00+00:00', '2019-01-01 00:00:00+00:00',
               '2019-01-02 00:00:00+00:00', '2019-01-03 00:00:00+00:00',
               '2019-01-04 00:00:00+00:00', '2019-01-05 00:00:00+00:00',
               '2019-01-06 00:00:00+00:00', '2019-01-07 00:00:00+00:00',
               '2019-01-08 00:00:00+00:00', '2019-01-09 00:00:00+00:00',
               ...
               '2023-12-21 00:00:00+00:00', '2023-12-22 00:00:00+00:00',
               '2023-12-23 00:00:00+00:00', '2023-12-24 00:00:00+00:00',
               '2023-12-25 00:00:00+00:00', '2023-12-26 00:00:00+00:00',
               '2023-12-27 00:00:00+00:00', '2023-12-28 00:00:00+00:00',
               '2023-12-29 00:00:00+00:00', '2023-12-30 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='date', length=1826, freq='D')

In [ ]:
df_daily.index

DatetimeIndex(['2018-12-31 00:00:00+00:00', '2019-01-01 00:00:00+00:00',
               '2019-01-02 00:00:00+00:00', '2019-01-03 00:00:00+00:00',
               '2019-01-04 00:00:00+00:00', '2019-01-05 00:00:00+00:00',
               '2019-01-06 00:00:00+00:00', '2019-01-07 00:00:00+00:00',
               '2019-01-08 00:00:00+00:00', '2019-01-09 00:00:00+00:00',
               ...
               '2023-12-21 00:00:00+00:00', '2023-12-22 00:00:00+00:00',
               '2023-12-23 00:00:00+00:00', '2023-12-24 00:00:00+00:00',
               '2023-12-25 00:00:00+00:00', '2023-12-26 00:00:00+00:00',
               '2023-12-27 00:00:00+00:00', '2023-12-28 00:00:00+00:00',
               '2023-12-29 00:00:00+00:00', '2023-12-30 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='date', length=1826, freq='D')

In [ ]:


# --- 5) Random Forest Test ---

feature_cols = [
    col for col in df_daily.columns
    if col not in ["profit", "cycles"]
]

X = df_daily[feature_cols].values
y = df_daily["profit"].values

n = len(df_daily)
split = int(n * 0.8)

X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)

print("Train R2:", r2_score(y_train, y_train_pred))
print("Val   R2:", r2_score(y_val, y_val_pred))
print("Val  MAE:", mean_absolute_error(y_val, y_val_pred))


Train R2: 0.9647697376602442
Val   R2: 0.030717821592573746
Val  MAE: 83.99822175961158
